# Manila Barangay Flood Risk Complete Pipeline

## 1. Project overview and methodology summary

**Thesis title focus:** Barangay-level flood risk priority classification for the City of Manila using a multi-criteria Disaster Prioritization Index (DPI) and supervised machine learning.

## Study unit and coverage
- **Unit of analysis:** barangay
- **Coverage:** all **897** barangays of the City of Manila (total enumeration, not sampling)
- If the final integrated dataset does not contain exactly 897 barangays, the notebook raises an error

## Pipeline stages (merged from three workflows)
1. **Shapefile-to-CSV** — Manila barangay boundaries ∩ LiPAD 5-year and 25-year flood hazard layers → flood area and flood percentage
2. **Merged dataset** — spatial flood CSV + PSA 2020 + PSA 2024 population + administrative attributes + elevation + engineered features
3. **Thesis ML** — CSI, Exposure, Vulnerability, entropy-weighted DPI, DPI risk classes, and RF / GB / MLP classification

## Updated methodology highlights
- Use **only** validated LiPAD **5-year** and **25-year** flood layers for DPI and ML (100-year excluded)
- Integrate **both** PSA 2020 and PSA 2024 populations
- Use **Population Change 2020–2024 (%)** (not annualized growth)
- Compute density with **PSA 2024** population
- Vulnerability uses density, population change, and reverse-normalized elevation (**Relative Population Density removed**)
- ML target is **DPI_Risk_Class** (index-derived priority classes, not historical flood outcomes)
- Preprocessing stays inside sklearn pipelines; model selection uses training CV only; held-out test is evaluated once

## Important interpretation note
Outputs are **planning-support** prototypes for barangay prioritization research. They are **not** official flood warnings, forecasts, or real-time inundation predictions.


## 2. Library imports and configuration

**Purpose:** Import required libraries and set global analysis constants.

**Input:** None (environment packages).

**Output:** Configured Python session (display options, random state, expected barangay count, CRS, tolerances).

**Why needed:** Reproducible thesis runs require a fixed random state, consistent numeric handling, and explicit study parameters (897 barangays; EPSG:32651 for area).


In [ ]:
from __future__ import annotations

import os
import re
import warnings
from pathlib import Path
from typing import Iterable, Optional

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

EXPECTED_BARANGAY_COUNT = 897
AREA_EPSG = 32651  # UTM Zone 51N for Manila area calculations
CITY_FILTER = "CITY OF MANILA"
RANDOM_STATE = 42
EPS = 1e-6

EXPECTED_PSA_2020_TOTAL = 1_846_513
EXPECTED_PSA_2024_TOTAL = 1_902_590
EXPECTED_POP_CHANGE_ABS = 56_077
EXPECTED_POP_CHANGE_PCT = 3.04

KEEP_ONLY_NUMBERED_BARANGAYS = True
EXPECTED_MIN_BARANGAY_NO = 1
EXPECTED_MAX_BARANGAY_NO = 905
APPLY_HAZARD_ATTRIBUTE_FILTER = False
HAZARD_FILTER_RULES = {}

print("Libraries imported.")
print(f"Expected barangays: {EXPECTED_BARANGAY_COUNT}")
print(f"Area CRS: EPSG:{AREA_EPSG}")
print(f"Random state: {RANDOM_STATE}")


## 3. Input path configuration

**Purpose:** Define all input and output paths in one place.

**Input:** User-editable paths to shapefiles, population files, administrative base data, optional elevation, and output directory.

**Output:** Resolved `Path` objects used by every later stage.

**Why needed:** After paths are set correctly, the notebook should run top-to-bottom without further interaction.

Update only this cell if your local folder layout differs. Shapefiles for 5-year and 25-year LiPAD layers plus Manila barangay boundaries are required for the spatial stage. If shapefiles are unavailable but a previously validated flood CSV exists, set `ALLOW_EXISTING_FLOOD_CSV_IF_SHAPEFILES_MISSING = True`.


In [ ]:
# ---------------------------------------------------------------------------
# USER CONFIGURATION — update paths if needed
# ---------------------------------------------------------------------------
BASE_DIR = Path("/workspace/src/ML-thesis")
SHP_DIR = BASE_DIR / "metro-manila-shp-to-csv"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADMIN_SHP = SHP_DIR / "phl_admin4.shp"
FLOOD_SHP_5YR = SHP_DIR / "MetroManila_Flood_5year.shp"
FLOOD_SHP_25YR = SHP_DIR / "MetroManila_Flood_25year.shp"
# 100-year layer is NOT used for DPI/ML. Path retained only as a commented validation note.
# FLOOD_SHP_100YR = SHP_DIR / "MetroManila_Flood_100year.shp"

PSA_2024_CSV = BASE_DIR / "population-dataset.csv"
PSA_2020_XLSX = BASE_DIR / "NCR.xlsx"
BASE_ADMIN_CSV = BASE_DIR / "manila_final_base_dataset_NEW.csv"
ELEVATION_CSV = BASE_DIR / "manila_barangay_elevation.csv"

EXISTING_FLOOD_CSV = SHP_DIR / "MetroManila_Barangay_Flood_Area_Validated.csv"
ALLOW_EXISTING_FLOOD_CSV_IF_SHAPEFILES_MISSING = True

OUT_FLOOD_CSV = OUTPUT_DIR / "MetroManila_Barangay_Flood_Area_Validated.csv"
OUT_ENRICHED_CSV = OUTPUT_DIR / "manila-city-flood-population_enriched_NEW.csv"
OUT_ENRICHED_XLSX = OUTPUT_DIR / "manila-city-flood-population_enriched_NEW.xlsx"
OUT_ML_READY_CSV = OUTPUT_DIR / "manila_ml_ready_dataset.csv"
OUT_MODEL_COMPARISON_CSV = OUTPUT_DIR / "model_comparison_results.csv"
OUT_PREDICTIONS_CSV = OUTPUT_DIR / "barangay_flood_risk_predictions.csv"
OUT_BEST_MODEL_JOBLIB = OUTPUT_DIR / "best_flood_risk_model_pipeline.joblib"

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR:", SHP_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PSA_2024_CSV exists:", PSA_2024_CSV.exists())
print("PSA_2020_XLSX exists:", PSA_2020_XLSX.exists())
print("BASE_ADMIN_CSV exists:", BASE_ADMIN_CSV.exists())
print("ELEVATION_CSV exists:", ELEVATION_CSV.exists())
print("Admin shapefile exists:", ADMIN_SHP.exists())
print("Flood 5yr shapefile exists:", FLOOD_SHP_5YR.exists())
print("Flood 25yr shapefile exists:", FLOOD_SHP_25YR.exists())
print("Existing flood CSV exists:", EXISTING_FLOOD_CSV.exists())


## Shared helper functions

**Purpose:** Centralize reusable operations used across spatial, merge, index, and ML stages.

**Input / output:** Pure utility functions (name cleaning, normalization, entropy weights, validation summaries).

**Why needed:** Thesis reproducibility and readability; repeated formulas (min-max, reverse min-max, entropy) must be applied consistently.


In [ ]:
def clean_barangay_name(x) -> str:
    """Standardize barangay names for merges (e.g., Brgy. 105 -> BARANGAY 105)."""
    x = str(x).upper().strip()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("BRGY.", "BARANGAY").replace("BRGY", "BARANGAY")
    return x


def clean_barangay_name_strict(x) -> str:
    """Strict merge key: remove spaces and hyphens (BARANGAY 202-A -> BARANGAY202A)."""
    return clean_barangay_name(x).replace("-", "").replace(" ", "")


def extract_barangay_number(name) -> Optional[int]:
    if pd.isna(name):
        return None
    match = re.search(r"Barangay\s+(\d+)", str(name), flags=re.IGNORECASE)
    return int(match.group(1)) if match else None


def extract_number(text) -> int:
    match = re.search(r"(\d+)", str(text))
    return int(match.group()) if match else 0


def is_numbered_barangay(name) -> bool:
    if pd.isna(name):
        return False
    return bool(re.fullmatch(r"Barangay\s+\d+(-[A-Z])?", str(name).strip(), flags=re.IGNORECASE))


def find_first_existing_column(df: pd.DataFrame, candidates: list, label: str) -> str:
    col = next((c for c in candidates if c in df.columns), None)
    if not col:
        raise ValueError(f"Could not find {label} column among {list(df.columns)}")
    return col


def keep_existing(cols: Iterable[str], df: pd.DataFrame) -> list:
    return [c for c in cols if c in df.columns]


def safe_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for col in cols:
        if col not in df.columns:
            continue
        if df[col].dtype == object:
            df[col] = (
                df[col].astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("%", "", regex=False)
            )
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def minmax_normalize(series: pd.Series) -> pd.Series:
    """Min-max normalization: higher raw value -> higher normalized risk contribution."""
    series = pd.to_numeric(series, errors="coerce")
    min_val, max_val = series.min(), series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(0.0, index=series.index)
    return (series - min_val) / (max_val - min_val)


def minmax_reverse(series: pd.Series) -> pd.Series:
    """Reverse min-max: lower raw value -> higher risk (used for elevation)."""
    series = pd.to_numeric(series, errors="coerce")
    min_val, max_val = series.min(), series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(0.0, index=series.index)
    return (max_val - series) / (max_val - min_val)


def entropy_weights(dataframe: pd.DataFrame):
    """Entropy weighting across component score columns; zeros handled safely for log."""
    x = dataframe.copy().replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    for col in x.columns:
        min_val, max_val = x[col].min(), x[col].max()
        x[col] = (x[col] - min_val) / (max_val - min_val) if max_val != min_val else 0.0

    col_sums = x.sum(axis=0)
    p = x.div(col_sums.replace(0, np.nan), axis=1).fillna(0)
    m = len(x)
    k = 1 / np.log(m) if m > 1 else 1.0
    p_log = p.replace(0, np.nan)
    entropy = (-k * (p * np.log(p_log)).sum(axis=0)).fillna(0)
    diversification = 1 - entropy
    if diversification.sum() == 0:
        weights = pd.Series(1 / len(diversification), index=diversification.index)
    else:
        weights = diversification / diversification.sum()
    return weights, entropy, diversification


def assert_exact_barangay_count(df: pd.DataFrame, stage: str, expected: int = EXPECTED_BARANGAY_COUNT) -> None:
    n = len(df)
    if n != expected:
        raise ValueError(
            f"[{stage}] Expected exactly {expected} barangays, found {n}. "
            "Check merges, filters, and input coverage for City of Manila."
        )
    print(f"[{stage}] Barangay count validated: {n}")


def validation_summary(title: str, checks: dict) -> pd.DataFrame:
    rows = [{"check": k, "count": int(v)} for k, v in checks.items()]
    report = pd.DataFrame(rows)
    print(f"\n=== {title} ===")
    print(report.to_string(index=False))
    return report


print("Helper functions defined.")


## 4. Shapefile loading and CRS validation

**Purpose:** Load Manila barangay boundaries and LiPAD 5-year / 25-year flood hazard shapefiles; inspect CRS and feature counts.

**Input:** `phl_admin4.shp`, `MetroManila_Flood_5year.shp`, `MetroManila_Flood_25year.shp`.

**Output:** GeoDataFrames for admin and flood layers, plus CRS diagnostics.

**Why needed:** Shapefile processing converts polygon hazard maps into barangay-level flood area and percentage indicators required by the thesis index. CRS must be known before area calculation.

**Why EPSG:32651:** Geographic CRS (degrees) cannot produce metric areas. UTM Zone 51N (EPSG:32651) is an appropriate projected CRS for Metro Manila, enabling area in square meters.

**Why only 5-year and 25-year:** These layers pass return-period consistency checks used in this thesis. The 100-year layer is excluded from DPI and ML because of known progression inconsistencies in the source hazard stack.


In [ ]:
def _import_geopandas():
    try:
        import geopandas as gpd
        return gpd
    except ImportError as exc:
        raise ImportError(
            "geopandas is required for the shapefile stage. Install with: pip install geopandas"
        ) from exc


def inspect_flood_layer(path: Path, rp: str):
    gpd = _import_geopandas()
    flood_raw = gpd.read_file(path)
    diagnostics = {
        "return_period": rp,
        "file": path.name,
        "crs": str(flood_raw.crs),
        "feature_count": len(flood_raw),
        "geometry_types": "; ".join(
            f"{k}:{v}" for k, v in flood_raw.geom_type.value_counts(dropna=False).to_dict().items()
        ),
        "columns": "; ".join(map(str, flood_raw.columns)),
    }
    return flood_raw, diagnostics


shapefiles_available = ADMIN_SHP.exists() and FLOOD_SHP_5YR.exists() and FLOOD_SHP_25YR.exists()
print("Shapefiles available for spatial stage:", shapefiles_available)

layer_diagnostics = []
admin_gdf = None
flood_layers = {}

if shapefiles_available:
    gpd = _import_geopandas()
    print("Loading admin shapefile:", ADMIN_SHP)
    admin_gdf = gpd.read_file(ADMIN_SHP)
    print("Admin CRS (before):", admin_gdf.crs)
    print("Admin feature count:", len(admin_gdf))
    print("Admin columns:", list(admin_gdf.columns))

    for rp, path in {"5yr": FLOOD_SHP_5YR, "25yr": FLOOD_SHP_25YR}.items():
        flood_raw, diag = inspect_flood_layer(path, rp)
        layer_diagnostics.append(diag)
        flood_layers[rp] = flood_raw
        print(f"\nFlood {rp}: CRS={flood_raw.crs}, features={len(flood_raw)}")
        print("Columns:", list(flood_raw.columns))
else:
    print(
        "Required shapefiles not found. Spatial intersection will be skipped if "
        "ALLOW_EXISTING_FLOOD_CSV_IF_SHAPEFILES_MISSING is True and a validated CSV exists."
    )


## 5. Geometry repair and preprocessing

**Purpose:** Filter City of Manila numbered barangays, repair invalid geometries, reproject to EPSG:32651, and compute barangay area (m²).

**Input:** Admin GeoDataFrame from the previous cell.

**Output:** Clean projected admin polygons with barangay area in square meters.

**Why needed:** Invalid polygons and geographic CRS would bias flood-area calculations. Numbered-barangay filtering removes non-barangay polygons so the study unit remains the barangay.


In [ ]:
def prepare_admin(admin):
    brgy_col = find_first_existing_column(
        admin, ["adm4_name", "ADM4_EN", "adm4_en", "barangay", "Barangay"], "barangay"
    )
    city_col = find_first_existing_column(
        admin, ["adm3_name", "ADM3_EN", "adm3_en", "city", "City"], "city"
    )

    admin = admin.loc[
        admin[city_col].astype(str).str.upper().str.strip() == CITY_FILTER.upper(),
        [brgy_col, "geometry"],
    ].copy()
    admin = admin.rename(columns={brgy_col: "barangay"})
    admin["barangay"] = admin["barangay"].astype(str).str.strip()
    print(f"Filtered to City of Manila: {len(admin)} polygons")

    special_rows = admin.loc[~admin["barangay"].apply(is_numbered_barangay)].copy()
    if not special_rows.empty:
        special_path = OUTPUT_DIR / "MetroManila_Special_NonBarangay_Rows.csv"
        special_rows.drop(columns="geometry").to_csv(special_path, index=False)
        print(f"Special/non-barangay rows: {len(special_rows)} -> {special_path}")

    if KEEP_ONLY_NUMBERED_BARANGAYS:
        before = len(admin)
        admin = admin.loc[admin["barangay"].apply(is_numbered_barangay)].copy()
        print(f"Kept numbered barangays: {before} -> {len(admin)}")

    print("CRS before reprojection:", admin.crs)
    admin = admin.to_crs(epsg=AREA_EPSG)
    print("CRS after reprojection:", admin.crs)

    invalid_before = int((~admin.geometry.is_valid).sum())
    admin["geometry"] = admin.geometry.buffer(0)
    invalid_after = int((~admin.geometry.is_valid).sum())
    print(f"Invalid geometries repaired (admin): {invalid_before} before buffer(0); {invalid_after} remain invalid")

    admin["brgy_area_sqm"] = admin.geometry.area
    if (admin["brgy_area_sqm"] <= 0).any():
        raise ValueError("Invalid negative or zero barangay areas found after projection.")

    return admin


if shapefiles_available:
    admin_gdf = prepare_admin(admin_gdf)
    manila_boundary = admin_gdf.dissolve()[["geometry"]].copy()
    print("Admin preprocessing complete. Sample:")
    print(admin_gdf[["barangay", "brgy_area_sqm"]].head())
else:
    manila_boundary = None
    print("Skipped admin preprocessing (shapefiles unavailable).")


## 6–7. Barangay–flood spatial intersection and flood percentage computation

**Purpose:** Intersect each flood hazard layer with barangay polygons; aggregate flood area; compute flood percentage for all barangays (zero-fill non-intersecting barangays).

**Conceptual formulas:**

`FloodArea_i,r = sum(area(Barangay_i intersect FloodPolygon_k,r))`

`FloodPCT_i,r = (FloodArea_i,r / BarangayArea_i) * 100`

where `r` is the 5-year or 25-year return period.

**Input:** Projected admin polygons and flood layers.

**Output:** Barangay table with flood area (m²) and flood percentage.

**Why needed:** Converts continuous hazard polygons into barangay-level indicators usable with population and vulnerability attributes.


In [ ]:
def apply_hazard_filter(flood):
    if not APPLY_HAZARD_ATTRIBUTE_FILTER:
        return flood, "No attribute filter applied"
    if not HAZARD_FILTER_RULES:
        return flood, "Filter enabled but no HAZARD_FILTER_RULES configured"
    for col, rule_func in HAZARD_FILTER_RULES.items():
        if col in flood.columns:
            before = len(flood)
            flood = flood.loc[rule_func(flood[col])].copy()
            return flood, f"Applied filter on '{col}': {before} -> {len(flood)}"
    return flood, "Filter enabled but no configured hazard column found"


def union_flood_geometries(flood, crs):
    gpd = _import_geopandas()
    if flood.empty:
        return gpd.GeoDataFrame(geometry=[], crs=crs)
    flood = flood.copy()
    flood["geometry"] = flood.geometry.buffer(0)
    flood = flood.loc[flood.geometry.notna() & ~flood.geometry.is_empty].copy()
    if flood.empty:
        return gpd.GeoDataFrame(geometry=[], crs=crs)
    try:
        union_geom = flood.geometry.union_all()
    except AttributeError:
        union_geom = flood.geometry.unary_union
    if union_geom is None or union_geom.is_empty:
        return gpd.GeoDataFrame(geometry=[], crs=crs)
    return gpd.GeoDataFrame(geometry=[union_geom], crs=crs)


def intersect_flood_with_barangays(admin, flood, rp: str, boundary):
    gpd = _import_geopandas()
    flood = flood.to_crs(admin.crs)
    flood = flood.loc[flood.geometry.notna() & ~flood.geometry.is_empty].copy()
    flood, note = apply_hazard_filter(flood)
    print(note)

    invalid_before = int((~flood.geometry.is_valid).sum())
    flood["geometry"] = flood.geometry.buffer(0)
    flood = flood.loc[flood.geometry.notna() & ~flood.geometry.is_empty].copy()
    print(f"Invalid flood geometries repaired ({rp}): {invalid_before}")

    flood = gpd.clip(flood, boundary)
    area_col = f"flood_area_{rp}_sqm"

    if flood.empty:
        return pd.DataFrame({"barangay": admin["barangay"], area_col: 0.0})

    flood_union = union_flood_geometries(flood, admin.crs)
    if flood_union.empty:
        return pd.DataFrame({"barangay": admin["barangay"], area_col: 0.0})

    joined = gpd.sjoin(
        flood_union,
        admin[["barangay", "geometry"]],
        how="inner",
        predicate="intersects",
    )
    if joined.empty:
        flood_by_brgy = pd.DataFrame(columns=["barangay", area_col])
    else:
        flood_geoms = joined.geometry.reset_index(drop=True)
        admin_geoms = admin.loc[joined["index_right"], "geometry"].reset_index(drop=True)
        inter_area = flood_geoms.intersection(admin_geoms).area
        joined = joined.reset_index(drop=True)
        joined["barangay"] = admin.loc[joined["index_right"], "barangay"].reset_index(drop=True).values
        joined[area_col] = inter_area.values
        flood_by_brgy = joined.groupby("barangay", as_index=False)[area_col].sum()
    return flood_by_brgy


flood_spatial_df = None

if shapefiles_available:
    result = pd.DataFrame({
        "barangay": admin_gdf["barangay"].values,
        "brgy_area_sqm": admin_gdf["brgy_area_sqm"].values,
    })

    for rp, flood in flood_layers.items():
        print(f"\nIntersecting {rp} flood layer...")
        flood_by_brgy = intersect_flood_with_barangays(admin_gdf, flood, rp, manila_boundary)
        result = result.merge(flood_by_brgy, on="barangay", how="left")
        area_col = f"flood_area_{rp}_sqm"
        pct_col = f"flood_pct_{rp}"
        result[area_col] = result[area_col].fillna(0.0)
        result[area_col] = result[area_col].clip(lower=0, upper=result["brgy_area_sqm"])
        result[pct_col] = (result[area_col] / result["brgy_area_sqm"] * 100).clip(0, 100)

    flood_spatial_df = result.sort_values("barangay").reset_index(drop=True)
    print("Spatial intersection complete. Shape:", flood_spatial_df.shape)
    print(flood_spatial_df.head())
else:
    print("Spatial intersection skipped (shapefiles unavailable).")


## 8. Validation of flood return-period consistency

**Purpose:** Enforce monotonicity between 5-year and 25-year flood percentages.

**Rule:** for each barangay, `Flood_PCT_25yr >= Flood_PCT_5yr`.

**Input:** Barangay flood area/percentage table.

**Output:** Validation flag `Return_Period_Valid_5_25` and diagnostics for inconsistent barangays.

**Why needed:** Inconsistent return-period progression undermines hazard escalation indicators used in CSI. Inconsistencies must be flagged, not used silently.


In [ ]:
def validate_return_period_consistency(df: pd.DataFrame) -> pd.DataFrame:
    required = ["flood_pct_5yr", "flood_pct_25yr"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing flood percentage columns for validation: {missing}")

    df = df.copy()
    df["Return_Period_Valid_5_25"] = ~(df["flood_pct_25yr"] + EPS < df["flood_pct_5yr"])
    df["flag_25yr_less_than_5yr"] = ~df["Return_Period_Valid_5_25"]

    n_bad = int((~df["Return_Period_Valid_5_25"]).sum())
    if n_bad > 0:
        print(f"WARNING: {n_bad} barangays violate Flood_PCT_25yr >= Flood_PCT_5yr")
        bad = df.loc[~df["Return_Period_Valid_5_25"], ["barangay", "flood_pct_5yr", "flood_pct_25yr"]]
        print(bad.head(30).to_string(index=False))
    else:
        print("Return-period consistency OK for all barangays (5yr <= 25yr).")

    for col in ["flood_pct_5yr", "flood_pct_25yr"]:
        outside = df[(df[col] < -EPS) | (df[col] > 100 + EPS)]
        if len(outside):
            raise ValueError(f"{col} has values outside 0-100 for {len(outside)} barangays.")

    return df


def load_existing_flood_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    rename_map = {
        "Barangay": "barangay",
        "Barangay Area Sqm": "brgy_area_sqm",
        "Flood Area 5yr Sqm": "flood_area_5yr_sqm",
        "Flood PCT 5yr": "flood_pct_5yr",
        "Flood Area 25yr Sqm": "flood_area_25yr_sqm",
        "Flood PCT 25yr": "flood_pct_25yr",
        "Barangay No": "barangay_no",
        "Flag 25yr Less than 5yr": "flag_25yr_less_than_5yr",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    drop_100 = [c for c in df.columns if "100" in c.lower()]
    if drop_100:
        print("Excluding 100-year columns from DPI/ML flood table:", drop_100)
        df = df.drop(columns=drop_100)
    needed = [
        "barangay", "brgy_area_sqm", "flood_area_5yr_sqm", "flood_pct_5yr",
        "flood_area_25yr_sqm", "flood_pct_25yr",
    ]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Existing flood CSV missing required columns: {missing}")
    df["barangay"] = df["barangay"].astype(str).str.strip()
    df = safe_numeric(df, needed[1:])
    print(f"Loaded existing flood CSV: {path} ({len(df)} rows)")
    return df


if flood_spatial_df is None:
    if ALLOW_EXISTING_FLOOD_CSV_IF_SHAPEFILES_MISSING and EXISTING_FLOOD_CSV.exists():
        flood_spatial_df = load_existing_flood_csv(EXISTING_FLOOD_CSV)
    else:
        raise FileNotFoundError(
            "Shapefiles missing and no existing validated flood CSV available. "
            "Update SHP_DIR / EXISTING_FLOOD_CSV in the configuration cell."
        )

flood_spatial_df = validate_return_period_consistency(flood_spatial_df)
if "barangay_no" not in flood_spatial_df.columns:
    flood_spatial_df["barangay_no"] = flood_spatial_df["barangay"].apply(extract_barangay_number)

print("Flood barangays loaded:", len(flood_spatial_df))


## 9. Export of spatial flood CSV

**Purpose:** Persist the validated barangay flood table for reproducibility and downstream merging.

**Input:** Validated flood spatial dataframe.

**Output:** `MetroManila_Barangay_Flood_Area_Validated.csv`

**Why needed:** Separates geospatial processing from demographic integration and provides an auditable intermediate product for the thesis appendix.


In [ ]:
export_flood = flood_spatial_df.copy()
export_cols = [
    "barangay", "brgy_area_sqm",
    "flood_area_5yr_sqm", "flood_pct_5yr",
    "flood_area_25yr_sqm", "flood_pct_25yr",
    "Return_Period_Valid_5_25", "flag_25yr_less_than_5yr", "barangay_no",
]
export_cols = [c for c in export_cols if c in export_flood.columns]
export_flood[export_cols].to_csv(OUT_FLOOD_CSV, index=False)
print(f"Exported spatial flood CSV: {OUT_FLOOD_CSV}")
print("Rows:", len(export_flood))
print(export_flood[export_cols].head())


## 10. Loading of PSA 2020 and PSA 2024 population datasets

**Purpose:** Load official PSA population snapshots for both census/reference years used in the thesis.

**Input:**
- PSA 2024: `population-dataset.csv`
- PSA 2020: `NCR.xlsx` if present; otherwise `bgypop2020` from `manila_final_base_dataset_NEW.csv`

**Output:** Clean population tables with `Population_2020` and `Population_2024`.

**Why both years are integrated:** Population Change 2020–2024 (%) requires both snapshots. Using only one year would omit the demographic-change vulnerability indicator required by the updated methodology.


In [ ]:
def load_psa_2024(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"PSA 2024 population file not found: {path}")
    df = pd.read_csv(path, header=None)
    df = df.iloc[:, :2].copy()
    df.columns = ["Barangay", "Population_2024"]
    df["Barangay"] = df["Barangay"].astype(str).str.strip()
    df = df[
        df["Barangay"].notna()
        & (df["Barangay"].str.strip() != "")
        & (df["Barangay"].str.lower() != "nan")
    ].copy()
    df["Population_2024"] = pd.to_numeric(df["Population_2024"], errors="coerce")
    df = df[df["Population_2024"].notna()].copy()
    numbered = df["Barangay"].apply(is_numbered_barangay)
    if numbered.any():
        before = len(df)
        df = df.loc[numbered].copy()
        print(f"PSA 2024 numbered-barangay filter: {before} -> {len(df)}")
    print(f"Loaded PSA 2024: {len(df)} barangays; total={df['Population_2024'].sum():,.0f}")
    return df


def load_psa_2020(xlsx_path: Path, base_csv: Path) -> pd.DataFrame:
    """Load PSA 2020 from NCR.xlsx when available; else from base admin bgypop2020."""
    if xlsx_path.exists():
        print(f"Loading PSA 2020 from {xlsx_path}")
        xl = pd.ExcelFile(xlsx_path)
        frames = []
        for sheet in xl.sheet_names:
            raw = pd.read_excel(xlsx_path, sheet_name=sheet)
            cols_lower = {c: str(c).lower() for c in raw.columns}
            brgy_cand = [c for c, lc in cols_lower.items() if any(k in lc for k in ["barangay", "brgy", "adm4"])]
            pop_cand = [
                c for c, lc in cols_lower.items()
                if ("2020" in lc and "pop" in lc) or lc in {"population", "pop", "total_population"}
            ]
            if brgy_cand and pop_cand:
                tmp = raw[[brgy_cand[0], pop_cand[0]]].copy()
                tmp.columns = ["Barangay", "Population_2020"]
                frames.append(tmp)
        if not frames:
            raise ValueError(f"Could not detect barangay/population columns in {xlsx_path}")
        df = pd.concat(frames, ignore_index=True)
        df["Barangay"] = df["Barangay"].astype(str).str.strip()
        df["Population_2020"] = pd.to_numeric(df["Population_2020"], errors="coerce")
        df = df[df["Population_2020"].notna()].drop_duplicates(subset=["Barangay"])
        print(f"Loaded PSA 2020 from NCR.xlsx: {len(df)} rows; total={df['Population_2020'].sum():,.0f}")
        return df

    if not base_csv.exists():
        raise FileNotFoundError(
            f"PSA 2020 source not found. Provide NCR.xlsx at {xlsx_path} "
            f"or base dataset with bgypop2020 at {base_csv}."
        )

    print(f"NCR.xlsx not found; using bgypop2020 from {base_csv}")
    base = pd.read_csv(base_csv)
    base.columns = base.columns.str.lower().str.strip().str.replace(" ", "_", regex=False)
    brgy_col = find_first_existing_column(base, ["brgy_name", "barangay", "barangay_name", "brgy"], "barangay")
    if "bgypop2020" not in base.columns:
        raise ValueError("Base dataset missing bgypop2020 (PSA 2020 population).")
    df = base[[brgy_col, "bgypop2020"]].copy()
    df.columns = ["Barangay", "Population_2020"]
    df["Barangay"] = df["Barangay"].astype(str).str.strip()
    df["Population_2020"] = pd.to_numeric(df["Population_2020"], errors="coerce")
    df = df[df["Population_2020"].notna()].drop_duplicates(subset=["Barangay"])
    print(f"Loaded PSA 2020 from base bgypop2020: {len(df)} rows; total={df['Population_2020'].sum():,.0f}")
    return df


df_pop_2024 = load_psa_2024(PSA_2024_CSV)
df_pop_2020 = load_psa_2020(PSA_2020_XLSX, BASE_ADMIN_CSV)


## 11. Barangay name/key standardization

**Purpose:** Create consistent merge keys across flood, population, and administrative tables.

**Input:** Raw barangay name strings from each source.

**Output:** `barangay_clean` and `barangay_key_strict` fields.

**Why needed:** Sources use inconsistent spacing, casing, and hyphenation (e.g., `Brgy. 202-A` vs `Barangay 202A`). Standardization is required for a complete 897-barangay match.


In [ ]:
def add_merge_keys(df: pd.DataFrame, barangay_col: str = "Barangay") -> pd.DataFrame:
    df = df.copy()
    df[barangay_col] = df[barangay_col].astype(str).str.strip()
    df["barangay_clean"] = df[barangay_col].apply(clean_barangay_name)
    df["barangay_key_strict"] = df[barangay_col].apply(clean_barangay_name_strict)
    return df


df_pop_2024 = add_merge_keys(df_pop_2024, "Barangay")
df_pop_2020 = add_merge_keys(df_pop_2020, "Barangay")

df_flood = flood_spatial_df.rename(columns={
    "barangay": "Barangay",
    "brgy_area_sqm": "Barangay_Area_Sqm",
    "flood_area_5yr_sqm": "Flood_Area_5yr_Sqm",
    "flood_pct_5yr": "Flood_PCT_5yr",
    "flood_area_25yr_sqm": "Flood_Area_25yr_Sqm",
    "flood_pct_25yr": "Flood_PCT_25yr",
    "barangay_no": "Barangay_No",
}).copy()
df_flood = add_merge_keys(df_flood, "Barangay")

print("Merge keys created for population and flood tables.")
print("2024 keys sample:", df_pop_2024[["Barangay", "barangay_key_strict"]].head(3).to_dict("records"))
print("Flood keys sample:", df_flood[["Barangay", "barangay_key_strict"]].head(3).to_dict("records"))


## 12. Population dataset validation and 897-barangay matching

**Purpose:** Merge PSA 2020 and PSA 2024 onto the flood barangay universe and validate complete coverage.

**Input:** Flood table (897 barangays) + PSA 2020 + PSA 2024.

**Output:** Matched barangay table with both population years; citywide total checks.

**Why needed:** The thesis uses total enumeration of Manila's 897 barangays. Missing population values would bias exposure and vulnerability scores.


In [ ]:
df = df_flood.merge(
    df_pop_2024[["barangay_key_strict", "Population_2024"]],
    on="barangay_key_strict",
    how="left",
)
df = df.merge(
    df_pop_2020[["barangay_key_strict", "Population_2020"]],
    on="barangay_key_strict",
    how="left",
)

print("After population merges:", len(df))
print("Missing Population_2024:", int(df["Population_2024"].isna().sum()))
print("Missing Population_2020:", int(df["Population_2020"].isna().sum()))

missing_2024 = df.loc[df["Population_2024"].isna(), ["Barangay", "barangay_key_strict"]]
missing_2020 = df.loc[df["Population_2020"].isna(), ["Barangay", "barangay_key_strict"]]
if len(missing_2024):
    print("Barangays missing PSA 2024:")
    print(missing_2024.head(20).to_string(index=False))
if len(missing_2020):
    print("Barangays missing PSA 2020:")
    print(missing_2020.head(20).to_string(index=False))

if df["Population_2024"].isna().any() or df["Population_2020"].isna().any():
    raise ValueError(
        "Population integration incomplete: one or more barangays lack Population_2020 or Population_2024."
    )

assert_exact_barangay_count(df, stage="population_match")

total_2020 = float(df["Population_2020"].sum())
total_2024 = float(df["Population_2024"].sum())
change_abs = total_2024 - total_2020
change_pct = (change_abs / total_2020) * 100 if total_2020 else np.nan

print(f"\nCitywide Population_2020: {total_2020:,.0f} (expected {EXPECTED_PSA_2020_TOTAL:,})")
print(f"Citywide Population_2024: {total_2024:,.0f} (expected {EXPECTED_PSA_2024_TOTAL:,})")
print(f"Absolute change: {change_abs:,.0f} (expected {EXPECTED_POP_CHANGE_ABS:,})")
print(f"Percentage change: {change_pct:.4f}% (expected ~{EXPECTED_POP_CHANGE_PCT}%)")

if abs(total_2020 - EXPECTED_PSA_2020_TOTAL) > 1:
    print("WARNING: PSA 2020 citywide total differs from expected reference.")
if abs(total_2024 - EXPECTED_PSA_2024_TOTAL) > 1:
    print("WARNING: PSA 2024 citywide total differs from expected reference.")
if abs(change_abs - EXPECTED_POP_CHANGE_ABS) > 1:
    print("WARNING: Absolute population change differs from expected reference.")


## 13. Integration of flood, population, administrative, area, and elevation data

**Purpose:** Attach administrative identifiers (PSGC, district/`city_name`), official area metadata, and elevation when available.

**Input:** Matched flood-population table + `manila_final_base_dataset_NEW.csv` + optional elevation CSV.

**Output:** Integrated analytical dataframe for feature engineering.

**Why needed:** District context features and elevation susceptibility require administrative joins beyond flood and population alone.


In [ ]:
if not BASE_ADMIN_CSV.exists():
    raise FileNotFoundError(f"Required administrative base dataset not found: {BASE_ADMIN_CSV}")

df_base = pd.read_csv(BASE_ADMIN_CSV)
df_base.columns = df_base.columns.str.lower().str.strip().str.replace(" ", "_", regex=False)
base_barangay_col = find_first_existing_column(
    df_base, ["brgy_name", "barangay", "barangay_name", "brgy"], "barangay"
)
df_base["barangay_key_strict"] = df_base[base_barangay_col].apply(clean_barangay_name_strict)

base_keep = keep_existing(
    [
        "barangay_key_strict",
        "psgc_10d",
        "reg_name",
        "city_name",
        "brgy_code",
        "bgyarea_sqkm",
        "geographic_level",
        "class",
        "urban_rural2015",
        "bgypop2020",
    ],
    df_base,
)
df_base_sel = df_base[base_keep].drop_duplicates(subset=["barangay_key_strict"])
df = df.merge(df_base_sel, on="barangay_key_strict", how="left")
print(
    "Admin metadata matched:",
    int(df["city_name"].notna().sum()) if "city_name" in df.columns else "city_name missing",
)
print(
    "Unmatched admin rows:",
    int(df["city_name"].isna().sum()) if "city_name" in df.columns else "n/a",
)

elevation_loaded = False
if ELEVATION_CSV.exists():
    df_elev = pd.read_csv(ELEVATION_CSV)
    elev_brgy = next((c for c in ["Barangay", "barangay", "brgy_name"] if c in df_elev.columns), None)
    elev_val = next(
        (
            c for c in [
                "Elevation", "Mean Elevation", "mean_elevation", "Avg Elevation", "avg_elevation",
            ]
            if c in df_elev.columns
        ),
        None,
    )
    if elev_brgy and elev_val:
        df_elev = df_elev.rename(columns={elev_val: "Elevation"})
        df_elev["barangay_key_strict"] = df_elev[elev_brgy].apply(clean_barangay_name_strict)
        df = df.merge(df_elev[["barangay_key_strict", "Elevation"]], on="barangay_key_strict", how="left")
        elevation_loaded = bool(df["Elevation"].notna().any())
        print(f"Elevation merged: {int(df['Elevation'].notna().sum())} barangays with values")
    else:
        print("Elevation file found but required columns not detected.")
else:
    print(
        "Elevation CSV not found. Vulnerability will use available indicators only "
        "(documented fallback from the original thesis workflow)."
    )

assert_exact_barangay_count(df, stage="integrated_admin_merge")
print("Integrated dataset shape:", df.shape)


## 14. Feature engineering

**Purpose:** Compute thesis indicators for hazard escalation, exposure, density, population change, and district context.

**Key definitions (updated methodology):**
- `Population_Change_2020_2024 = Population_2024 - Population_2020`
- `Population_Change_2020_2024_Pct = ((Population_2024 - Population_2020) / Population_2020) * 100` — **not annualized**
- `Barangay_Area_Hectares = Barangay_Area_Sqm / 10000`
- `Population_Density_per_Hectare = Population_2024 / Barangay_Area_Hectares`
- Affected population uses **PSA 2024** only
- `Flood_Escalation_5to25` and `Flood_Severity_Increase_5to25_Sqm` flagged if negative

**Why Population Change (not growth rate):** The indicator is a four-year observed change between official PSA snapshots, not an annualized demographic rate.

**Why density uses 2024:** Exposure and current vulnerability should reflect the more recent population snapshot.

**Why Relative Population Density is removed:** The updated vulnerability specification uses absolute density, population change, and elevation only. Relative density is excluded from vulnerability and from the final ML feature list.


In [ ]:
df = safe_numeric(
    df,
    [
        "Population_2020", "Population_2024", "Barangay_Area_Sqm",
        "Flood_Area_5yr_Sqm", "Flood_PCT_5yr", "Flood_Area_25yr_Sqm", "Flood_PCT_25yr",
        "bgyarea_sqkm", "Elevation",
    ],
)

df["Population_Change_2020_2024"] = df["Population_2024"] - df["Population_2020"]
df["Population_Change_2020_2024_Pct"] = np.where(
    df["Population_2020"] > 0,
    (df["Population_Change_2020_2024"] / df["Population_2020"]) * 100,
    np.nan,
)

df["Barangay_Area_Hectares"] = df["Barangay_Area_Sqm"] / 10_000
if (df["Barangay_Area_Hectares"] <= 0).any():
    raise ValueError("Non-positive barangay area in hectares detected.")
df["Population_Density_per_Hectare"] = df["Population_2024"] / df["Barangay_Area_Hectares"]

df["Affected_Population_5yr"] = df["Population_2024"] * (df["Flood_PCT_5yr"] / 100)
df["Affected_Population_25yr"] = df["Population_2024"] * (df["Flood_PCT_25yr"] / 100)
df["Affected_Population_Increase_5to25"] = (
    df["Affected_Population_25yr"] - df["Affected_Population_5yr"]
)

df["Flood_Escalation_5to25"] = df["Flood_PCT_25yr"] - df["Flood_PCT_5yr"]
df["Flood_Severity_Increase_5to25_Sqm"] = df["Flood_Area_25yr_Sqm"] - df["Flood_Area_5yr_Sqm"]

neg_esc = df["Flood_Escalation_5to25"] < -EPS
neg_sev = df["Flood_Severity_Increase_5to25_Sqm"] < -EPS
if neg_esc.any() or neg_sev.any():
    print(
        f"WARNING: Negative escalation/severity detected "
        f"(escalation={int(neg_esc.sum())}, severity_sqm={int(neg_sev.sum())})."
    )
    print(
        df.loc[
            neg_esc | neg_sev,
            [
                "Barangay", "Flood_PCT_5yr", "Flood_PCT_25yr",
                "Flood_Escalation_5to25", "Flood_Severity_Increase_5to25_Sqm",
            ],
        ].head(20)
    )

if "bgyarea_sqkm" in df.columns:
    df["Base_Area_Sqm"] = df["bgyarea_sqkm"] * 1_000_000
    df["Area_Difference_Sqm"] = df["Barangay_Area_Sqm"] - df["Base_Area_Sqm"]
    df["Area_Difference_Pct"] = np.where(
        df["Base_Area_Sqm"] > 0,
        (df["Area_Difference_Sqm"] / df["Base_Area_Sqm"]) * 100,
        np.nan,
    )

if "city_name" in df.columns:
    df["District_Avg_Flood_PCT_5yr"] = df.groupby("city_name")["Flood_PCT_5yr"].transform("mean")
    df["District_Avg_Flood_PCT_25yr"] = df.groupby("city_name")["Flood_PCT_25yr"].transform("mean")
    df["District_Avg_Population_Density_per_Hectare"] = (
        df.groupby("city_name")["Population_Density_per_Hectare"].transform("mean")
    )
    df["Relative_Flood_Risk_25yr_vs_District"] = (
        df["Flood_PCT_25yr"] - df["District_Avg_Flood_PCT_25yr"]
    )
    # NOTE: Relative Population Density is intentionally NOT computed for vulnerability/ML.

df["Sort_Key"] = df["Barangay"].apply(extract_number)
df = df.sort_values(["Sort_Key", "Barangay"]).drop(columns=["Sort_Key"]).reset_index(drop=True)

print("Feature engineering complete.")
print(
    df[[
        "Barangay", "Population_2020", "Population_2024", "Population_Change_2020_2024_Pct",
        "Population_Density_per_Hectare", "Flood_Escalation_5to25",
        "Affected_Population_5yr", "Affected_Population_25yr",
    ]].head()
)


## 15. Export of enriched analytical dataset

**Purpose:** Save the full integrated barangay table for documentation and downstream DPI/ML.

**Input:** Feature-engineered dataframe.

**Output:**
- `manila-city-flood-population_enriched_NEW.csv`
- `manila-city-flood-population_enriched_NEW.xlsx`

**Why needed:** Provides a human-readable and machine-readable archive of all analytical fields before index construction.


In [ ]:
assert_exact_barangay_count(df, stage="enriched_export")

geom_cols = [c for c in df.columns if str(c).lower() in {"geometry", "geom"}]
if geom_cols:
    df = df.drop(columns=geom_cols)

df.to_csv(OUT_ENRICHED_CSV, index=False)
try:
    df.to_excel(OUT_ENRICHED_XLSX, index=False)
    print(f"Exported Excel: {OUT_ENRICHED_XLSX}")
except Exception as exc:
    print(f"Excel export skipped ({exc})")

print(f"Exported enriched CSV: {OUT_ENRICHED_CSV}")
print("Final enriched shape:", df.shape)
print("Missing value summary (top 15):")
print(df.isna().sum().sort_values(ascending=False).head(15))


## 16. Preparation of ML-ready dataset scaffold

**Purpose:** Identify identifier columns versus candidate analytic fields ahead of CSI/DPI computation.

**Input:** Enriched dataset.

**Output:** Working dataframe `df_ml` for index and modeling stages.

**Why needed:** Separates identifiers from numeric indicators and prepares a clean table for leakage-controlled modeling.


In [ ]:
df_ml = df.copy()
df_ml.columns = df_ml.columns.astype(str).str.strip()

id_cols = keep_existing(
    [
        "Barangay", "Barangay_No", "psgc_10d", "city_name", "brgy_code",
        "Population_2020", "Population_2024",
    ],
    df_ml,
)
print("Identifier / reference columns:", id_cols)
print("Working ML dataframe shape:", df_ml.shape)
assert_exact_barangay_count(df_ml, stage="ml_scaffold")


## 17. CSI computation

**Purpose:** Compute the Composite Susceptibility Index (hazard component) from validated 5-year and 25-year indicators only.

**Indicators (equal-weighted after min-max normalization):**
- `Flood_PCT_5yr`
- `Flood_PCT_25yr`
- `Flood_Escalation_5to25`
- `Flood_Severity_Increase_5to25_Sqm`

`CSI = mean(normalized CSI indicators)`

**Why CSI is separate:** DPI follows a multi-criteria structure where hazard, exposure, and vulnerability are computed independently before entropy weighting. Separating CSI keeps the hazard construct interpretable.

**Why the 100-year layer is excluded:** Return-period consistency issues in the 100-year source layer would distort CSI and downstream DPI classes.


In [ ]:
csi_inputs = [
    "Flood_PCT_5yr",
    "Flood_PCT_25yr",
    "Flood_Escalation_5to25",
    "Flood_Severity_Increase_5to25_Sqm",
]
missing = [c for c in csi_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing CSI inputs: {missing}")

df_ml = safe_numeric(df_ml, csi_inputs)
df_ml["norm_Flood_PCT_5yr"] = minmax_normalize(df_ml["Flood_PCT_5yr"])
df_ml["norm_Flood_PCT_25yr"] = minmax_normalize(df_ml["Flood_PCT_25yr"])
df_ml["norm_Flood_Escalation_5to25"] = minmax_normalize(df_ml["Flood_Escalation_5to25"])
df_ml["norm_Flood_Severity_Increase_5to25_Sqm"] = minmax_normalize(
    df_ml["Flood_Severity_Increase_5to25_Sqm"]
)

df_ml["CSI"] = df_ml[[
    "norm_Flood_PCT_5yr",
    "norm_Flood_PCT_25yr",
    "norm_Flood_Escalation_5to25",
    "norm_Flood_Severity_Increase_5to25_Sqm",
]].mean(axis=1)

print("CSI computed. Summary:")
print(df_ml["CSI"].describe())


## 18. Exposure score computation

**Purpose:** Quantify population potentially affected under 5-year and 25-year flood scenarios using PSA 2024.

**Indicators (equal-weighted after normalization):**
- `Affected_Population_5yr`
- `Affected_Population_25yr`
- `Affected_Population_Increase_5to25`

`Exposure = mean(normalized exposure indicators)`

**Why separate from CSI:** Exposure captures who is affected, not only how much land is flooded.


In [ ]:
exposure_inputs = [
    "Affected_Population_5yr",
    "Affected_Population_25yr",
    "Affected_Population_Increase_5to25",
]
missing = [c for c in exposure_inputs if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing Exposure inputs: {missing}")

df_ml = safe_numeric(df_ml, exposure_inputs)
df_ml["norm_Affected_Population_5yr"] = minmax_normalize(df_ml["Affected_Population_5yr"])
df_ml["norm_Affected_Population_25yr"] = minmax_normalize(df_ml["Affected_Population_25yr"])
df_ml["norm_Affected_Population_Increase_5to25"] = minmax_normalize(
    df_ml["Affected_Population_Increase_5to25"]
)
df_ml["Exposure"] = df_ml[[
    "norm_Affected_Population_5yr",
    "norm_Affected_Population_25yr",
    "norm_Affected_Population_Increase_5to25",
]].mean(axis=1)

print("Exposure computed. Summary:")
print(df_ml["Exposure"].describe())


## 19. Vulnerability score computation

**Purpose:** Compute demographic/physical susceptibility using the updated three-indicator specification.

**Indicators:**
1. `Population_Density_per_Hectare` (min-max)
2. `Population_Change_2020_2024_Pct` (min-max)
3. reverse-normalized `Elevation` (lower elevation → higher susceptibility)

`Vulnerability_i = (norm(Density_i) + norm(PopChange%_i) + norm_reverse(Elevation_i)) / 3`

**Why elevation is reverse-normalized:** Lower-lying barangays are generally more susceptible to inundation.

**Why Relative Population Density is removed:** It is no longer part of the thesis vulnerability construct and must not enter vulnerability or final ML features.

**Elevation fallback:** The original project documented a fallback when elevation is unavailable. If elevation is missing here, vulnerability is computed from the two available indicators and explicitly marked as a limitation.


In [ ]:
vuln_raw = ["Population_Density_per_Hectare", "Population_Change_2020_2024_Pct"]
missing = [c for c in vuln_raw if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing vulnerability inputs: {missing}")

df_ml = safe_numeric(df_ml, vuln_raw + keep_existing(["Elevation"], df_ml))
df_ml["norm_Population_Density_per_Hectare"] = minmax_normalize(df_ml["Population_Density_per_Hectare"])
df_ml["norm_Population_Change_2020_2024_Pct"] = minmax_normalize(df_ml["Population_Change_2020_2024_Pct"])

vulnerability_norm_cols = [
    "norm_Population_Density_per_Hectare",
    "norm_Population_Change_2020_2024_Pct",
]
elevation_used = False
if "Elevation" in df_ml.columns and df_ml["Elevation"].notna().any():
    if df_ml["Elevation"].isna().any():
        raise ValueError(
            "Elevation is partially missing. Provide complete elevation for all 897 barangays "
            "or remove the elevation column to use the documented two-indicator fallback."
        )
    df_ml["norm_Elevation_reverse"] = minmax_reverse(df_ml["Elevation"])
    vulnerability_norm_cols.append("norm_Elevation_reverse")
    elevation_used = True
    print("Elevation included in Vulnerability (reverse-normalized).")
else:
    print(
        "LIMITATION: Elevation unavailable. Vulnerability uses only "
        "Population_Density_per_Hectare and Population_Change_2020_2024_Pct "
        "(documented fallback from the original project)."
    )

df_ml["Vulnerability"] = df_ml[vulnerability_norm_cols].mean(axis=1)
df_ml["Vulnerability_Elevation_Included"] = elevation_used

print("Vulnerability components:", vulnerability_norm_cols)
print(df_ml["Vulnerability"].describe())


## 20. Entropy weighting

**Purpose:** Derive objective weights for the three high-level DPI components (CSI, Exposure, Vulnerability).

**Input:** Component score columns.

**Output:** Entropy values, degree of diversification, and entropy weights (printed).

**Why entropy weighting across component scores:** The thesis weights the three constructs by information content across barangays rather than assigning arbitrary fixed weights. Raw indicators are not entropy-weighted individually in the final method.


In [ ]:
component_cols = ["CSI", "Exposure", "Vulnerability"]
entropy_w, entropy_values, diversification_values = entropy_weights(df_ml[component_cols])

weights_table = pd.DataFrame({
    "Component": entropy_w.index,
    "Entropy_Value": entropy_values.values,
    "Diversification_Value": diversification_values.values,
    "Entropy_Weight": entropy_w.values,
})
print("Entropy-derived component weights:")
print(weights_table.to_string(index=False))
print("Weight sum:", float(entropy_w.sum()))
weights_table.to_csv(OUTPUT_DIR / "entropy_component_weights.csv", index=False)


## 21. DPI computation and scaling

**Purpose:** Combine weighted components into the Disaster Prioritization Index and rescale to 0–100.

`DPI = w_CSI * CSI + w_Exposure * Exposure + w_Vulnerability * Vulnerability`

`DPI_Scaled = minmax(DPI) * 100`


In [ ]:
df_ml["DPI"] = (
    entropy_w["CSI"] * df_ml["CSI"]
    + entropy_w["Exposure"] * df_ml["Exposure"]
    + entropy_w["Vulnerability"] * df_ml["Vulnerability"]
)
df_ml["DPI_Scaled"] = minmax_normalize(df_ml["DPI"]) * 100

print("DPI computed and scaled to 0-100.")
print(df_ml[["Barangay", "CSI", "Exposure", "Vulnerability", "DPI", "DPI_Scaled"]].head())
print(df_ml["DPI_Scaled"].describe())


## 22. DPI class generation using tertiles

**Purpose:** Create ordinal priority classes Low / Medium / High from scaled DPI using tertiles (`qcut`).

**Output column:** `DPI_Risk_Class`

**Why DPI-derived classes are ML targets:** Externally observed flood-impact labels are not available in this dataset. Models learn to approximate the deterministic prioritization index classes for methodological comparison — **not** historical flood-event outcomes and **not** real-time flood prediction.

If duplicate bin edges occur, `duplicates='drop'` is used and the approach is reported.


In [ ]:
try:
    df_ml["DPI_Risk_Class"] = pd.qcut(
        df_ml["DPI_Scaled"],
        q=3,
        labels=["Low", "Medium", "High"],
        duplicates="drop",
    )
    qcut_note = "Tertile classification via pd.qcut on DPI_Scaled (duplicates='drop')."
except ValueError as exc:
    ranks = df_ml["DPI_Scaled"].rank(method="first")
    df_ml["DPI_Risk_Class"] = pd.qcut(ranks, q=3, labels=["Low", "Medium", "High"])
    qcut_note = (
        f"qcut on raw DPI_Scaled failed ({exc}); used rank-based qcut to force three classes."
    )

print(qcut_note)
print("DPI_Risk_Class distribution:")
print(df_ml["DPI_Risk_Class"].value_counts().to_string())
print("Class proportions (%):")
print((df_ml["DPI_Risk_Class"].value_counts(normalize=True) * 100).round(2).to_string())


## 23. Leakage-free ML feature selection

**Purpose:** Build the predictor matrix while excluding all target- and index-derived fields.

**Excluded (non-exhaustive reminder):** CSI, Exposure, Vulnerability, DPI, DPI_Scaled, DPI_Risk_Class, normalized component intermediates, identifiers, validation flags, geometry, 100-year fields, Relative Population Density, Area_Difference_Pct (QC only).

**Recommended predictors (included when present):**
Flood percentages/escalation/severity, Population_2024, affected population fields, density, population-change %, elevation, and district context flood/density averages plus relative flood risk vs district.

**Why leakage prevention is necessary:** Including DPI components or the class label as predictors would trivially recover the target and invalidate model evaluation as a test of learning from raw/engineered inputs.


In [ ]:
target_col = "DPI_Risk_Class"

candidate_features = [
    "Flood_PCT_5yr",
    "Flood_PCT_25yr",
    "Flood_Escalation_5to25",
    "Flood_Severity_Increase_5to25_Sqm",
    "Population_2024",
    "Affected_Population_5yr",
    "Affected_Population_25yr",
    "Affected_Population_Increase_5to25",
    "Population_Density_per_Hectare",
    "Population_Change_2020_2024_Pct",
    "Elevation",
    "District_Avg_Flood_PCT_5yr",
    "District_Avg_Flood_PCT_25yr",
    "District_Avg_Population_Density_per_Hectare",
    "Relative_Flood_Risk_25yr_vs_District",
]

leakage_or_forbidden = {
    "CSI", "Hazard_Score", "Hazard Score", "CSI Score",
    "Exposure", "Exposure_Score", "Exposure Score",
    "Vulnerability", "Vulnerability_Score", "Vulnerability Score",
    "DPI", "DPI_Scaled", "DPI Score", "DPI Score 0to100",
    "DPI_Risk_Class", "DPI Risk Class",
    "Entropy_Weight", "Entropy_Value", "Diversification_Value",
    "Barangay", "Barangay_No", "psgc_10d", "brgy_code", "city_name",
    "barangay_clean", "barangay_key_strict",
    "Return_Period_Valid_5_25", "flag_25yr_less_than_5yr",
    "geometry", "Relative_Population_Density", "Relative Pop Density vs District",
    "Relative Pop Density 2024 vs District",
    "Area_Difference_Pct", "Area Difference Pct",
    "Population_2020",
}

ml_features = [c for c in candidate_features if c in df_ml.columns and c not in leakage_or_forbidden]
ml_features = list(dict.fromkeys(ml_features))

if not ml_features:
    raise ValueError("No valid ML features remain after leakage filtering.")

bad = [c for c in ml_features if c in leakage_or_forbidden]
if bad:
    raise ValueError(f"Leakage/forbidden features selected: {bad}")

print("Leakage-free ML features:")
for c in ml_features:
    print(" -", c)
print("Feature count:", len(ml_features))

ml_ready_cols = keep_existing(["Barangay", "Barangay_No", "psgc_10d", "city_name"], df_ml) + ml_features + [target_col]
df_ml_ready = df_ml[ml_ready_cols].copy()
df_ml_ready.to_csv(OUT_ML_READY_CSV, index=False)
print(f"Exported ML-ready dataset: {OUT_ML_READY_CSV} shape={df_ml_ready.shape}")


## 24. Train-test split

**Purpose:** Create an 80/20 stratified split by `DPI_Risk_Class` with `random_state=42`.

**Why both split and CV are used:** The held-out test set estimates final generalization once. Cross-validation on the **training set only** supports model selection without touching the test set.


In [ ]:
df_model = df_ml[ml_features + [target_col]].dropna(subset=[target_col]).copy()
X = df_model[ml_features]
y = df_model[target_col].astype(str)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Class encoding:")
for name, code_ in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"  {name} -> {code_}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=RANDOM_STATE, stratify=y_encoded
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Train class distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test class distribution:", dict(zip(*np.unique(y_test, return_counts=True))))


## 25. Model pipelines

**Purpose:** Define Random Forest, Gradient Boosting, and MLP pipelines with preprocessing inside the pipeline.

- Trees: `SimpleImputer(median) -> Classifier`
- MLP: `SimpleImputer(median) -> StandardScaler -> MLPClassifier`

**Why pipelines:** Imputation/scaling must fit on training folds only. Fitting on the full dataset before splitting would leak test information.


In [ ]:
base_models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
    "MLP": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            random_state=RANDOM_STATE,
            max_iter=1000,
            early_stopping=True,
        )),
    ]),
}

param_grids = {
    "Random Forest": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [None, 12],
        "model__min_samples_leaf": [1, 2],
    },
    "Gradient Boosting": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
    },
    "MLP": {
        "model__hidden_layer_sizes": [(64, 32), (32,)],
        "model__alpha": [1e-4, 1e-3],
    },
}

print("Model pipelines defined:", list(base_models.keys()))


## 26. Cross-validation and hyperparameter tuning

**Purpose:** Stratified 5-fold CV on the **training set** with primary selection metric **F1-macro**, using identical folds across models via a shared `StratifiedKFold`.

**Why F1-macro:** Treats Low/Medium/High classes equally, which is appropriate for priority-class balance assessment.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
tuned_models = {}
best_params = {}

for name, pipe in base_models.items():
    print(f"\nTuning {name} with GridSearchCV (training set only)...")
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    best_params[name] = search.best_params_

    cv_rows.append({
        "Model": name,
        "CV_F1_Macro_Mean": float(search.best_score_),
        "CV_F1_Macro_Std": float(search.cv_results_["std_test_score"][search.best_index_]),
        "Best_Params": str(search.best_params_),
    })
    print(f"Best CV F1-macro: {search.best_score_:.4f}")
    print("Best params:", search.best_params_)

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_F1_Macro_Mean", ascending=False)
print("\nCross-validation model selection summary:")
print(cv_results_df.to_string(index=False))


## 27. Final held-out test evaluation

**Purpose:** Select the best model by training CV F1-macro, ensure it is refit on the full training set (GridSearchCV `refit=True`), then evaluate **once** on the held-out test set.

**Why evaluate the selected model only once on the test set:** Repeated test-set peeking would bias the final generalization estimate. Test metrics are for final reporting, not model selection.

Reported metrics include Accuracy, Balanced Accuracy, per-class Precision/Recall/F1, F1-Macro, F1-Weighted, ROC-AUC OvR Macro (when probabilities exist), Confusion Matrix, and Classification Report.


In [ ]:
def evaluate_model(name, model, X_te, y_te, label_encoder):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te) if hasattr(model, "predict_proba") else None
    try:
        roc = (
            roc_auc_score(y_te, y_proba, multi_class="ovr", average="macro")
            if y_proba is not None else np.nan
        )
    except Exception:
        roc = np.nan

    precision, recall, f1, support = precision_recall_fscore_support(
        y_te, y_pred, labels=list(range(len(label_encoder.classes_))), zero_division=0
    )
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_te, y_pred),
        "F1_Macro": f1_score(y_te, y_pred, average="macro"),
        "F1_Weighted": f1_score(y_te, y_pred, average="weighted"),
        "ROC_AUC_OVR_Macro": roc,
    }
    for i, cls in enumerate(label_encoder.classes_):
        metrics[f"Precision_{cls}"] = precision[i]
        metrics[f"Recall_{cls}"] = recall[i]
        metrics[f"F1_{cls}"] = f1[i]
        metrics[f"Support_{cls}"] = support[i]

    cm = confusion_matrix(y_te, y_pred)
    report = classification_report(y_te, y_pred, target_names=label_encoder.classes_)
    return metrics, cm, report, y_pred


best_model_name = cv_results_df.iloc[0]["Model"]
best_model = tuned_models[best_model_name]
print("Best model by training CV F1-macro:", best_model_name)

test_rows = []
confusion_matrices = {}
classification_reports = {}

for name, model in tuned_models.items():
    print(f"\n==== Held-out test evaluation: {name} ====")
    metrics, cm, report, _ = evaluate_model(name, model, X_test, y_test, label_encoder)
    test_rows.append(metrics)
    confusion_matrices[name] = cm
    classification_reports[name] = report
    print(report)
    print("Confusion matrix:\n", cm)

test_results_df = pd.DataFrame(test_rows)
print("\nSelected best model test metrics:")
print(test_results_df.loc[test_results_df["Model"] == best_model_name].T)


## 28. Model comparison table

**Purpose:** Combine training CV selection metrics and held-out test metrics for all three models into one comparison table.

**Output:** `model_comparison_results.csv`


In [ ]:
comparison_df = cv_results_df.merge(test_results_df, on="Model", how="left")
comparison_df = comparison_df.sort_values(["CV_F1_Macro_Mean", "F1_Macro"], ascending=False)
comparison_df.to_csv(OUT_MODEL_COMPARISON_CSV, index=False)
print("Model comparison table:")
print(comparison_df.to_string(index=False))
print(f"Saved: {OUT_MODEL_COMPARISON_CSV}")


## 29. Feature importance extraction

**Purpose:** Extract impurity-based feature importance for Random Forest and Gradient Boosting.

**Interpretation:** Values are **model-based importance** for approximating DPI-derived classes. They do **not** imply causal drivers of real-world flooding.

If the selected best model is MLP, tree-model importances are still reported; standard MLP has no direct impurity-based importance.


In [ ]:
importance_frames = {}
for name in ["Random Forest", "Gradient Boosting"]:
    model = tuned_models[name]
    clf = model.named_steps["model"]
    if hasattr(clf, "feature_importances_"):
        imp_df = pd.DataFrame({
            "Feature": ml_features,
            "Importance": clf.feature_importances_,
            "Model": name,
        }).sort_values("Importance", ascending=False)
        importance_frames[name] = imp_df
        out_path = OUTPUT_DIR / f"feature_importance_{name.lower().replace(' ', '_')}.csv"
        imp_df.to_csv(out_path, index=False)
        print(f"\nFeature importance ({name}) — model-based only:")
        print(imp_df.to_string(index=False))

if best_model_name == "MLP":
    print(
        "\nNote: Best model is MLP. Standard MLPClassifier does not provide "
        "impurity-based feature importance; tree-model importances above are reported for interpretation support only."
    )


## 30. Barangay-level prediction and ranking output

**Purpose:** Apply the selected best model to all barangays; attach predicted class, confidence, DPI scores, and a DPI-based rank.

**Output:** `barangay_flood_risk_predictions.csv`

**Interpretation reminder:** These are planning-support priority outputs derived from an index-based prototype, not official flood warnings.


In [ ]:
X_all = df_ml[ml_features].copy()
df_ml["ML_Predicted_Class_Encoded"] = best_model.predict(X_all)
df_ml["ML_Predicted_Risk_Class"] = label_encoder.inverse_transform(df_ml["ML_Predicted_Class_Encoded"])
if hasattr(best_model, "predict_proba"):
    df_ml["ML_Prediction_Confidence"] = best_model.predict_proba(X_all).max(axis=1)
else:
    df_ml["ML_Prediction_Confidence"] = np.nan
df_ml["Best_Model"] = best_model_name
df_ml["DPI_Rank"] = df_ml["DPI_Scaled"].rank(ascending=False, method="min").astype(int)

pred_cols = keep_existing(
    [
        "Barangay", "Barangay_No", "psgc_10d", "city_name",
        "Population_2020", "Population_2024",
        "Flood_PCT_5yr", "Flood_PCT_25yr",
        "CSI", "Exposure", "Vulnerability",
        "DPI", "DPI_Scaled", "DPI_Risk_Class", "DPI_Rank",
        "ML_Predicted_Risk_Class", "ML_Prediction_Confidence", "Best_Model",
        "Vulnerability_Elevation_Included",
    ],
    df_ml,
)
predictions_out = df_ml[pred_cols].sort_values("DPI_Rank").reset_index(drop=True)
predictions_out.to_csv(OUT_PREDICTIONS_CSV, index=False)
print(f"Saved barangay predictions: {OUT_PREDICTIONS_CSV}")
print(predictions_out.head(10))


## 31. Export of final model and outputs

**Purpose:** Serialize the best pipeline and confirm all required thesis outputs exist.

**Outputs:**
1. `MetroManila_Barangay_Flood_Area_Validated.csv`
2. `manila-city-flood-population_enriched_NEW.csv`
3. `manila-city-flood-population_enriched_NEW.xlsx`
4. `manila_ml_ready_dataset.csv`
5. `model_comparison_results.csv`
6. `barangay_flood_risk_predictions.csv`
7. `best_flood_risk_model_pipeline.joblib`


In [ ]:
joblib.dump(
    {
        "pipeline": best_model,
        "label_encoder": label_encoder,
        "features": ml_features,
        "best_model_name": best_model_name,
        "best_params": best_params.get(best_model_name),
        "random_state": RANDOM_STATE,
    },
    OUT_BEST_MODEL_JOBLIB,
)

required_outputs = [
    OUT_FLOOD_CSV,
    OUT_ENRICHED_CSV,
    OUT_ENRICHED_XLSX,
    OUT_ML_READY_CSV,
    OUT_MODEL_COMPARISON_CSV,
    OUT_PREDICTIONS_CSV,
    OUT_BEST_MODEL_JOBLIB,
]

print("\nOutput file checklist:")
for path in required_outputs:
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {path}")

missing_outputs = [p for p in required_outputs if not p.exists()]
if missing_outputs:
    critical = [p for p in missing_outputs if p.suffix.lower() != ".xlsx"]
    if critical:
        raise FileNotFoundError(f"Missing required outputs: {critical}")
    print("WARNING: Excel output missing (non-critical if openpyxl unavailable).")

print("\nBest model:", best_model_name)
print("Feature count:", len(ml_features))
print("Barangay count:", len(df_ml))


## 32. Final reproducibility and limitation notes

**Reproducibility**
- Fixed `random_state=42` for split, CV folds, and model estimators
- Preprocessing confined to sklearn pipelines / GridSearchCV folds
- Model selection uses training CV F1-macro only; held-out test evaluated after selection
- Intermediate and final outputs written to the configured `OUTPUT_DIR`

**Methodological limitations**
- LiPAD flood layers are modeled hazard surfaces, not observed event inundation maps
- The 100-year layer is excluded from DPI/ML due to return-period consistency concerns
- DPI risk classes are index-derived proxy labels, not ground-truth flood impacts
- Supervised metrics measure agreement with the deterministic DPI classification, not real-world forecast skill
- If elevation is unavailable, vulnerability uses the documented two-indicator fallback and this should be stated in the thesis limitations
- Population Change 2020–2024 (%) is a four-year observed change, not an annualized growth rate
- District features depend on the administrative `city_name` district partition present in the base dataset

**Use statement**
Prototype outputs support barangay-level planning prioritization research for the City of Manila. They must not be interpreted as official flood warnings, evacuation orders, or real-time flood predictions.


In [ ]:
print("=" * 72)
print("Manila flood risk complete pipeline finished.")
print(f"Barangays: {len(df_ml)} (expected {EXPECTED_BARANGAY_COUNT})")
print(f"Best model: {best_model_name}")
print(f"Outputs directory: {OUTPUT_DIR}")
print("=" * 72)
